# Electromagnetic shower

*nuclear · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/nuclear/em_shower)

A high-energy electron hitting a block of lead starts a cascade: electrons radiate photons, photons turn into electron–positron pairs, and the energy is halved at each step until it drops to the critical energy. The Heitler model is simulated as a branching process on tensors — each particle a row, each split two new rows — and predicts the shower maximum $N_{max} \approx E_0/E_c$ at depth $t_{max} \approx \ln(E_0/E_c)/\ln 2$.

**Vector form**

$$e^\pm \to e^\pm + \gamma,\quad \gamma \to e^+e^-,\quad E \to \tfrac{E}{2}\ \text{per } X_0,\qquad N(t) = 2^t\ \text{until}\ \tfrac{E_0}{2^t} \approx E_c\ \Rightarrow\ N_{max} \approx \frac{E_0}{E_c},\ \ t_{max} \approx \frac{\ln(E_0/E_c)}{\ln 2}$$

**Index (tensor) form**

$$(E_p, s_p) \to \big(\tfrac{E_p}{2}, 1\big),\ \big(\tfrac{E_p}{2}, 1 - s_p\big)\ \text{ if } \tfrac{E_p}{2} \ge E_c,\qquad N(t) = \textstyle\sum_p 1,\qquad \sum_p E_p(t) + E_{dep}(t) = E_0$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $E_p,\ s_p$ | `state['energy'], state['kind']` | (P,) | energy (MeV) and type (1 = e±, 0 = γ) of every particle in flight; P changes as the shower grows |
| $p_{int}$ | `interaction_probability(kind, dt)` | (P,) | stochastic mode: $1 - e^{-\Delta t/\lambda}$, $\lambda = X_0$ (e±) or $\tfrac97 X_0$ (γ) |
| $E_p/2 \ge E_c$ | `can_split(energy, e_crit)` | (P,) | boolean mask: who multiplies, the rest deposit their energy |
| $(\tfrac{E_p}{2}, 1), (\tfrac{E_p}{2}, 1-s_p)$ | `children(energy[split], kind[split])` | (2S,) | e → e + γ and γ → e + e, as two concatenated halves |
| $N(t)$ | `result['counts'].sum(dim=1)` | (T,) | particles in flight at each depth |
| $E_{dep}(t)$ | `result['deposited']` | (T,) | energy dropped by particles below the critical energy |

Depth is in radiation lengths $X_0$ (0.56 cm in lead) and $E_c$ is the critical energy, about 7.4 MeV in lead. The doubling stops when the daughters would fall below $E_c$, so the maximum of about $E_0/E_c$ particles is reached after about $\log_2(E_0/E_c)$ radiation lengths: the depth of a shower grows only logarithmically with energy, which is why calorimeters can be compact. In the fixed-length mode every particle splits after exactly one $X_0$, so a whole generation moves in step and the count climbs like a staircase. The stochastic mode (the dashed curve shows the other mode for comparison) draws each interaction point at random (photons travel $\tfrac97 X_0$ on average): the generations smear out in depth, giving the lower, broader and longer profile of real showers, while the energy bookkeeping stays exact. The lateral spread is only illustrative (an angle kick $\propto E_c/E$). Simplifications of the Heitler model: equal energy sharing, no ionisation loss above $E_c$, a particle below threshold deposits all its energy at once, and no Compton scattering or photoelectric effect. The random seed is fixed.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/nuclear/em_shower/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import heitler_n_max, heitler_t_max


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    e0, e_crit = params["e0"] * 1000.0, params["e_crit"]  # MeV
    n_max, t_max = heitler_n_max(e0, e_crit), heitler_t_max(e0, e_crit)
    steps_per_x0 = 20
    depths = tensors.depth_axis(t_max + 8.0, steps_per_x0, dev)  # (T,) in X0

    stochastic = params["mode"] == "stochastic"
    result = solve.shower(tensors.primary(e0, dev), depths, steps_per_x0, e_crit, stochastic)
    other = solve.shower(tensors.primary(e0, dev), depths, steps_per_x0, e_crit, stochastic=not stochastic)
    reference = other["counts"].sum(dim=1)  # (T,) the other splitting mode, for comparison

    total = result["counts"].sum(dim=1)
    k = int(total.argmax())
    balance = (result["in_flight"] + result["deposited"] - e0).abs().max().item()
    print(f"E0 = {e0:.0f} MeV, Ec = {e_crit:.1f} MeV  ({params['mode']} splitting)")
    print(f"Heitler estimate: N_max ≈ E0/Ec = {n_max:.0f}   t_max ≈ ln(E0/Ec)/ln2 = {t_max:.2f} X0")
    print(f"simulated:        N_max = {total[k].item()}   at depth t = {depths[k].item():.2f} X0")
    print(f"particles created: {len(result['tracks'])}   max |Σ E_in_flight + E_deposited − E0| = {balance:.1e} MeV")
    other_label = "fixed length (split every X₀)" if stochastic else "stochastic length"
    return plot.animate(depths, result, reference, other_label, n_max, t_max, out_dir / "em_shower.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/nuclear/em_shower/equation.py
r"""Physics → tensor translation.

The Heitler model of an electromagnetic shower: a high-energy electron radiates a photon (bremsstrahlung),
a photon converts into an electron–positron pair, and each time the energy is shared equally:

    e^± → e^± + γ,        γ → e^+ + e^-,        E_{child} = E_{parent} / 2.

Depth t is measured in radiation lengths X_0. In the simplest version every particle splits after one X_0,
so after t steps there are 2^t particles of energy E_0 / 2^t. Multiplication stops once the daughters
would fall below the critical energy E_c (where ionisation losses beat radiation), giving

    N_{max} ≈ E_0 / E_c,        t_{max} ≈ \ln(E_0/E_c) / \ln 2.

On tensors, particles p carry energy E_p and type s_p (1 = e^±, 0 = γ); a split replaces each parent by two
children with energies E_p/2 and types (1, 1 − s_p). Energy is conserved at every depth:

    Σ_p E_p(t) + E_{dep}(t) = E_0.

Stochastic variant: an e^± interacts with probability 1 − e^{−Δt/λ} per step Δt with λ = X_0, a photon with
λ = 9/7 X_0 (its mean free path for pair production).
"""

import math

import torch

PHOTON, ELECTRON = 0, 1


def interaction_probability(kind: torch.Tensor, dt: float) -> torch.Tensor:
    r"""p_p = 1 − e^{−Δt/λ_p},  λ = 1 X_0 for e^±, 9/7 X_0 for γ.   kind: (P,) → (P,)"""
    mean_free_path = torch.where(kind == ELECTRON, 1.0, 9.0 / 7.0)
    return 1.0 - torch.exp(-dt / mean_free_path)


def can_split(energy: torch.Tensor, e_crit: float) -> torch.Tensor:
    r"""A particle multiplies only while its daughters stay above the critical energy: E_p/2 ≥ E_c."""
    return energy / 2 >= e_crit


def children(energy: torch.Tensor, kind: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    r"""Two daughters per parent: energies (E/2, E/2), types (e^±, 1 − s): e → e + γ, γ → e + e.

    energy, kind: (P,) → energies (2P,), kinds (2P,)  [first all first daughters, then all second ones]
    """
    half = energy / 2
    first = torch.full_like(kind, ELECTRON)
    second = 1 - kind
    return torch.cat([half, half]), torch.cat([first, second])


def heitler_n_max(e0: float, e_crit: float) -> float:
    r"""N_{max} ≈ E_0 / E_c."""
    return e0 / e_crit


def heitler_t_max(e0: float, e_crit: float) -> float:
    r"""t_{max} ≈ \ln(E_0/E_c) / \ln 2  (in X_0)."""
    return math.log(e0 / e_crit) / math.log(2.0)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/nuclear/em_shower/tensors.py
"""Build the concrete tensors: the primary particle and the depth axis."""

import torch

from .equation import ELECTRON


def primary(e0: float, device: torch.device) -> dict[str, torch.Tensor]:
    """One electron of energy E0 (MeV) entering at depth 0 on the axis. Each entry has shape (P,) = (1,).

    energy E_p, kind s_p, lateral position x_p and angle a_p (in X0 per X0), the depth z0_p and lateral
    position x0_p where it was born (for drawing its track), and its birth step (for the fixed-length mode).
    """
    as_float = {"dtype": torch.float64, "device": device}
    return {
        "energy": torch.tensor([e0], **as_float),
        "kind": torch.tensor([ELECTRON], device=device),
        "x": torch.zeros(1, **as_float),
        "angle": torch.zeros(1, **as_float),
        "z0": torch.zeros(1, **as_float),
        "x0": torch.zeros(1, **as_float),
        "born": torch.zeros(1, dtype=torch.long, device=device),
    }


def depth_axis(t_end: float, steps_per_x0: int, device: torch.device) -> torch.Tensor:
    """t_k = k / steps_per_x0 radiation lengths: shape (T,)."""
    n = int(round(t_end * steps_per_x0))
    return torch.arange(n + 1, dtype=torch.float64, device=device) / steps_per_x0


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/nuclear/em_shower/solve.py
"""Branching simulation: every particle is a row of a few tensors; splits replace rows by two new ones."""

import torch

from problems._common.progress import steps

from .equation import can_split, children, interaction_probability


def shower(
    state: dict[str, torch.Tensor],
    depths: torch.Tensor,
    steps_per_x0: int,
    e_crit: float,
    stochastic: bool,
    spread: float = 0.5,
    seed: int = 0,
) -> dict[str, torch.Tensor]:
    """Follow the cascade through the depth grid.

    Returns counts (T, 2) of [photons, e±] in flight, energy in flight (T,), deposited energy (T,), and the
    track segments (M, 5) = (z_start, x_start, z_end, x_end, kind) of every particle for drawing.
    """
    gen = torch.Generator(device=depths.device).manual_seed(seed)
    dt = 1.0 / steps_per_x0
    deposited = torch.zeros((), dtype=torch.float64, device=depths.device)
    counts, in_flight, dep, tracks = [], [], [], []

    def record():
        kind = state["kind"]
        counts.append(torch.stack([(kind == 0).sum(), (kind == 1).sum()]))
        in_flight.append(state["energy"].sum())
        dep.append(deposited.clone())

    record()
    for step in steps(len(depths) - 1, "depth steps"):
        z = depths[step + 1]
        state["x"] = state["x"] + state["angle"] * dt  # fly one step forward
        if stochastic:
            u = torch.rand(state["energy"].shape, generator=gen, device=depths.device, dtype=torch.float64)
            hit = u < interaction_probability(state["kind"], dt)
        else:
            hit = (step + 1 - state["born"]) >= steps_per_x0  # exactly one X0 after birth
        split = hit & can_split(state["energy"], e_crit)
        absorbed = hit & ~split

        tracks.append(_segments(state, hit, z))  # every particle that interacted ends its track here
        deposited = deposited + state["energy"][absorbed].sum()  # too soft to multiply: energy stays here

        energy, kind = children(state["energy"][split], state["kind"][split])  # (2S,) each
        n_split = int(split.sum())
        if stochastic:
            direction = torch.randn(2 * n_split, generator=gen, device=depths.device, dtype=torch.float64)
        else:
            direction = torch.cat([torch.ones(n_split), -torch.ones(n_split)]).to(depths)  # symmetric tree
        new = {
            "energy": energy,
            "kind": kind,
            "x": state["x"][split].repeat(2),
            "angle": state["angle"][split].repeat(2) + spread * e_crit / energy * direction,
            "z0": z.expand(2 * n_split),
            "x0": state["x"][split].repeat(2),
            "born": torch.full((2 * n_split,), step + 1, device=depths.device),
        }
        keep = ~hit
        state = {key: torch.cat([state[key][keep], new[key]]) for key in state}
        record()

    everyone = torch.ones_like(state["energy"], dtype=torch.bool)
    tracks.append(_segments(state, everyone, depths[-1]))  # particles still flying at the end
    return {
        "counts": torch.stack(counts),
        "in_flight": torch.stack(in_flight),
        "deposited": torch.stack(dep),
        "tracks": torch.cat(tracks),
    }


def _segments(state: dict[str, torch.Tensor], mask: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
    """(z_start, x_start, z_end, x_end, kind) of the masked particles: shape (m, 5)."""
    m = int(mask.sum())
    return torch.stack(
        [state["z0"][mask], state["x0"][mask], z.expand(m), state["x"][mask], state["kind"][mask].to(z.dtype)], dim=1
    )


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/nuclear/em_shower/plot.py
"""Top: the shower growing in depth (e± solid, γ dashed). Bottom: particle count against depth."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.collections import LineCollection

from problems._common.plotting import frame_indices, save_gif


def animate(
    depths: torch.Tensor,
    result: dict[str, torch.Tensor],
    reference: torch.Tensor,
    reference_label: str,
    n_max: float,
    t_max: float,
    path: Path,
) -> Path:
    t = depths.cpu().numpy()
    counts = result["counts"].cpu().numpy()  # (T, 2) photons, e±
    total = counts.sum(axis=1)
    ref = reference.cpu().numpy()  # (T,) total count with the other splitting mode
    z0, x0, z1, x1, kind = result["tracks"].cpu().numpy().T

    fig, (ax_s, ax_n) = plt.subplots(2, 1, figsize=(8, 5), sharex=True, gridspec_kw={"height_ratios": [1.1, 1]})
    lateral = np.percentile(np.abs(np.concatenate([x0, x1])), 99.5) * 1.1 + 1e-3
    ax_s.set_xlim(0, t[-1])
    ax_s.set_ylim(-lateral, lateral)
    ax_s.set_ylabel("lateral (X₀)")
    ax_s.grid(False)
    electrons = LineCollection([], colors="#2563eb", linewidths=0.6, alpha=0.7)
    photons = LineCollection([], colors="#f59e0b", linewidths=0.6, alpha=0.7, linestyles=(0, (2, 2)))
    ax_s.add_collection(photons)
    ax_s.add_collection(electrons)
    ax_s.plot([], [], color="#2563eb", label="e±")
    ax_s.plot([], [], color="#f59e0b", ls="--", label="γ")
    ax_s.legend(loc="upper left", fontsize=8)
    front_s = ax_s.axvline(0, color="#111827", lw=1)
    title = ax_s.set_title(" ", fontsize=9)  # reserve room before tight_layout

    ax_n.plot(t, ref, color="#9ca3af", lw=1.5, ls="--", label=reference_label)
    ax_n.axhline(n_max, color="#f59e0b", lw=1, ls=":")
    ax_n.text(t[-1], n_max, r"$E_0/E_c$ ", ha="right", va="bottom", fontsize=8, color="#b45309")
    ax_n.axvline(t_max, color="#f59e0b", lw=1, ls=":")
    ax_n.text(t_max, 0.02 * n_max, r" $\ln(E_0/E_c)/\ln 2$", fontsize=8, color="#b45309")
    (line_tot,) = ax_n.plot([], [], color="#111827", lw=2, label="all particles")
    (line_e,) = ax_n.plot([], [], color="#2563eb", lw=1.2, label="e±")
    (line_g,) = ax_n.plot([], [], color="#f59e0b", lw=1.2, label="γ")
    ax_n.set_ylim(0, max(total.max(), ref.max(), n_max) * 1.12)
    ax_n.set_xlabel("depth t (radiation lengths X₀)")
    ax_n.set_ylabel("particles in flight")
    ax_n.legend(loc="upper right", fontsize=7)

    def clipped(front: float, which: np.ndarray) -> np.ndarray:
        sel = which & (z0 < front)
        end = np.minimum(z1[sel], front)
        frac = (end - z0[sel]) / np.maximum(z1[sel] - z0[sel], 1e-12)
        x_end = x0[sel] + frac * (x1[sel] - x0[sel])
        return np.stack([np.column_stack([z0[sel], x0[sel]]), np.column_stack([end, x_end])], axis=1)

    def update(step: int):
        front = t[step]
        electrons.set_segments(clipped(front, kind == 1))
        photons.set_segments(clipped(front, kind == 0))
        front_s.set_xdata([front])
        line_tot.set_data(t[: step + 1], total[: step + 1])
        line_e.set_data(t[: step + 1], counts[: step + 1, 1])
        line_g.set_data(t[: step + 1], counts[: step + 1, 0])
        title.set_text(f"depth {front:.1f} X₀   particles in flight: {total[step]}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 80), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'e0': 10.0,
    'e_crit': 7.4,
    'mode': 'fixed'
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.nuclear.em_shower")]:
    del sys.modules[name]
main = importlib.import_module("problems.nuclear.em_shower.main")
problem_dir = Path("problems/nuclear/em_shower")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
